## Problem:
**A binary classifier on tabular data (10,000 rows) shows steadily falling training loss, but validation accuracy stays near 55% and fluctuates between epochs.**

In [1]:
# Provided training code:

# X = StandardScaler().fit_transform(X)
# X_tr, X_val = X[:8000], X[8000:]
# y_tr, y_val = y[:8000], y[8000:]
# np.random.shuffle(X_tr)

# model = nn.Sequential(nn.Linear(40, 128), nn.ReLU(), nn.Dropout(0.5),
#               nn.BatchNorm1d(128), nn.Linear(128, 2))
# opt = torch.optim.Adam(model.parameters(), lr=1e-3)
# scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=1, gamma=0.9)

# for epoch in range(50):
#     model.train()
#     for xb, yb in train_loader:
#         loss = loss_fn(model(xb), yb)
#         opt.zero_grad(); loss.backward(); opt.step()
#         scheduler.step()
#     val_logits = model(torch.tensor(X_val))
#     print(epoch, accuracy(val_logits, y_val))

## Required analysis:

- Identify every bug you can find.
- Explain which symptom each bug causes.
- Provide corrected code or a description of each fix.
- Describe quick experiments that would reveal pipeline problems without access to the code.

## Solutions


## milestone 1
## bugs:
1. Scaling before the train/validation split: Optimistic or unreliable validation results.
2. np.random.shuffle(X_tr) alone: Features and targets no longer match; the model cannot learn the intended mapping.
3. No shuffle of paired training examples: Unstable gradients or poor generalization.
4. BatchNorm placed after Dropout: A train/evaluation distribution mismatch can reduce validation performance.
5. No validation-label alignment checks: Misleadingly low validation accuracy despite falling training loss.
6. Potential incorrect target format: Errors or invalid loss computation if labels are encoded incorrectly.
7. No explicit device and dtype management: Dtype errors, unnecessary conversions, or device mismatches.
8. Possible class imbalance or incorrect accuracy interpretation: Apparently reasonable accuracy despite little predictive value.


## Corrected Code

In [2]:
# import random
# import numpy as np
# import torch
# import torch.nn as nn

# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import (
#     accuracy_score,
#     balanced_accuracy_score,
#     f1_score,
#     roc_auc_score,
# )
# from torch.utils.data import TensorDataset, DataLoader

# # 1. Reproducibility
# seed = 42
# random.seed(seed)
# np.random.seed(seed)
# torch.manual_seed(seed)

# # 2. Split BEFORE fitting preprocessing
# X_tr, X_val, y_tr, y_val = train_test_split(
#     X, y,
#     test_size=0.20,
#     random_state=seed,
#     stratify=y,
# )

# # 3. Fit scaler only on training features
# scaler = StandardScaler()
# X_tr = scaler.fit_transform(X_tr).astype(np.float32)
# X_val = scaler.transform(X_val).astype(np.float32)

# # 4. Ensure correct tensor types
# X_tr = torch.tensor(X_tr, dtype=torch.float32)
# X_val = torch.tensor(X_val, dtype=torch.float32)
# y_tr = torch.tensor(
#     np.asarray(y_tr), dtype=torch.long
# ).reshape(-1)
# y_val = torch.tensor(
#     np.asarray(y_val), dtype=torch.long
# ).reshape(-1)

# # 5. Shuffle examples together, not features independently
# train_ds = TensorDataset(X_tr, y_tr)
# train_loader = DataLoader(
#     train_ds, batch_size=128, shuffle=True
# )

# model = nn.Sequential(
#     nn.Linear(40, 128),
#     nn.BatchNorm1d(128),
#     nn.ReLU(),
#     nn.Dropout(0.3),
#     nn.Linear(128, 2),
# )

# loss_fn = nn.CrossEntropyLoss()
# opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# # Decay once per epoch
# scheduler = torch.optim.lr_scheduler.StepLR(
#     opt, step_size=10, gamma=0.5
# )

# for epoch in range(50):
#     model.train()
#     total_loss = 0.0
#     total_examples = 0

#     for xb, yb in train_loader:
#         opt.zero_grad(set_to_none=True)

#         logits = model(xb)
#         loss = loss_fn(logits, yb)

#         loss.backward()
#         opt.step()

#         total_loss += loss.item() * len(xb)
#         total_examples += len(xb)

#     scheduler.step()

#     # Evaluation: no dropout and fixed BatchNorm statistics
#     model.eval()
#     with torch.inference_mode():
#         val_logits = model(X_val)
#         val_pred = val_logits.argmax(dim=1)

#         val_acc = accuracy_score(y_val.numpy(), val_pred.numpy())
#         val_bal_acc = balanced_accuracy_score(
#             y_val.numpy(), val_pred.numpy()
#         )
#         val_f1 = f1_score(
#             y_val.numpy(), val_pred.numpy(), zero_division=0
#         )

#     avg_loss = total_loss / total_examples

#     print(
#         f"Epoch {epoch + 1:02d} | "
#         f"Loss: {avg_loss:.4f} | "
#         f"Val Acc: {val_acc:.4f} | "
#         f"Balanced Acc: {val_bal_acc:.4f} | "
#         f"F1: {val_f1:.4f} | "
#         f"LR: {opt.param_groups[0]['lr']:.2e}"
#     )

## quick experiments

If the pipeline is a black box, investigate it using observable inputs, outputs, and controlled experiments.
1. Majority-class baseline: Predict the most frequent training class for every validation row. If this achieves about 55%, the current accuracy may be no better than the simplest baseline.
2. Label permutation test: Randomly permute training labels and rerun training. Performance should generally collapse toward chance or the relevant baseline. Strong performance suggests leakage, duplicate examples, or a flawed evaluation.
3. Small-subset overfit test: Train on 32–128 examples. A sufficiently expressive model should usually fit a clean, learnable subset. Failure points toward data-label misalignment, preprocessing errors, or optimization issues.
4. Repeated-run test: Train several times with different random seeds. Large variation can indicate unstable optimization, small validation samples, or a fragile pipeline.
5. Feature-label alignment test: If the interface permits it, compare performance after jointly permuting rows of features and labels versus independently permuting one side. Joint permutation should preserve the dataset; independent permutation should destroy the signal.
6. Learning-rate test: Compare the current schedule with a fixed learning rate. A dramatic improvement suggests the original schedule decayed too aggressively.
7. Validation-mode test: Compare repeated evaluation of the same validation set. Predictions should be deterministic in evaluation mode, apart from nondeterminism in certain hardware operations.
8. Distribution-shift test: Compare feature distributions and class proportions across training and validation periods. Large differences may indicate temporal drift or an inappropriate split.